# `function_name`: the question it answers, in one line

> **Template:** how to start a gallery notebook from this file.
> 1. Copy it to `gallery/<family>/<function>.ipynb`. The family is the last part of
>    the function's module: `easy_viz.plots.comparison` → `comparison`.
> 2. Replace every heading and every *Template* note like this one. Keep the seven
>    sections, in this order.
> 3. Add a row for the notebook to `gallery/README.md`.
> 4. Execute it only with the build script, which also renders the static images:
>    `uv run --isolated --python 3.14 python scripts/build_gallery.py --only <function>`

One or two sentences for the reader: what this chart is for, and the story this
notebook tells with the showcase data.

Every table of the showcase data becomes a SQL view, so each chart below starts from
a query, the way it would in a data warehouse.

> **Template:** keep this setup cell unchanged. Query with
> `db.sql("SELECT ... FROM orders JOIN customers USING (customer_id) ...")`. easy-viz
> takes the query result itself, or the table you get by ending the query with
> `.pl()` (polars), `.df()` (pandas) or `.to_arrow_table()` (pyarrow). Vary them
> across notebooks. Not `.arrow()`: in DuckDB 1.5 it returns a stream of record
> batches, not a table.

In [ ]:
from pathlib import Path

import duckdb

import easy_viz as ev

db = duckdb.connect()
for csv in sorted(Path("../data").glob("*.csv")):
    db.sql(f"CREATE VIEW {csv.stem} AS FROM '{csv.as_posix()}'")

## The question

> **Template:** in two or three sentences, the business question this chart answers
> and which planted pattern of the showcase data it will uncover (name the pattern as
> `gallery/data/INSIGHTS.md` does). Then query the data the chart needs.

In [ ]:
# The query behind the chart, written as an analyst would write it. For example:
# sales = db.sql("""
#     SELECT region, SUM(revenue) AS revenue
#     FROM orders JOIN customers USING (customer_id)
#     GROUP BY region
#     ORDER BY revenue DESC
# """).pl()

## Bare call

> **Template:** one sentence on what the default chart already shows, then the
> one-line call with no options at all. This is the "simple charts take a few
> options" promise: the reader should see a finished chart from one line.

In [ ]:
# The default call: the table and the required columns, nothing else.
# ev.function_name(sales, "region", "revenue")

## Parameter tour

> **Template:** every keyword option of the function, grouped by purpose, one
> `###` sub-heading per group in this order (skip a group the function doesn't
> have): data and aggregation, ordering, labels, colour, reference lines, layout,
> theme. For each option:
> - one or two sentences: what it does and what changes on the chart;
> - its default and accepted values, quoted from the docstring word for word;
> - a code cell that uses it as `option=...` and shows the figure.
>
> A test fails if any keyword option never appears as `option=` in a code cell.

### Data and aggregation

In [ ]:
# One option per cell, so each figure shows exactly one change. For example:
# raw = db.sql("SELECT region, revenue FROM orders JOIN customers USING (customer_id)")
# ev.function_name(raw, "region", "revenue", agg="sum")

## Combinations

> **Template:** two or three realistic scenarios, each a question an analyst would
> actually ask, answered by several options together. Open each with the scenario in
> one sentence ("Which regions are above target this quarter?"), then the call. This
> is the "professional charts take more options, never more code" promise.

In [ ]:
# A realistic scenario: several options in one call. For example:
# ev.function_name(
#     sales,
#     "region",
#     "revenue",
#     sort_by="revenue",
#     top_n=5,
#     value_format=",.0f",
#     title="Revenue by region",
# )

## Limits

> **Template:** what the function refuses, and why that protects the reader.
> - Every pair of options that conflict: call them together and print the real error
>   message, so the reader sees exactly what they would get.
> - Every edge case the docstring promises (empty data, a single category, nulls,
>   ...): show what happens.

In [ ]:
# Catch the error and print it: an uncaught error would stop the notebook.
# try:
#     ev.function_name(sales, "region", ["revenue", "cost"], color="segment")
# except ValueError as error:
#     print(error)

## Insight story

> **Template:** the product value made visible. Start from the bare chart, where the
> planted pattern is hidden, and add one option per step until the chart exposes it.
> Before each step, one sentence on what the reader should notice and why the next
> option helps. End with a sentence stating the insight as a business finding.
>
> **README image:** to use the final chart in a README, tag its cell
> `readme:<name>`: add `"tags": ["readme:<name>"]` to the cell's metadata (in
> JupyterLab: property inspector, *Cell Tags*). The name uses lowercase letters,
> digits, `-` and `_`, and the cell must show exactly one figure. The build script
> then writes `docs/images/readme/<name>.png`; link that file from the README.

In [ ]:
# One step per cell, each adding the option that reveals a little more.
# The last step's cell carries the tag "readme:<name>" in its metadata if the README
# uses it. For example:
# ev.function_name(sales, "region", "revenue", color="segment", highlight="North")

## Escape hatch

> **Template:** what no option covers is finished on the returned figure, which is a
> plain Plotly `go.Figure`. Show one such edit that targets a piece of the chart by
> its trace role (the roles are listed in the function's docstring, section
> *Trace roles*), and say in one sentence why a reader might want it.

In [ ]:
# Edit the returned figure, selecting traces by their role. For example:
# fig = ev.function_name(sales, "region", "revenue")
# fig.update_traces(
#     line_dash="dot",
#     selector=lambda trace: (trace.meta or {}).get("role") == "reference",
# )